# 01 — The maths, properly typeset

Every equation this project rests on, rendered. Open this in VSCode (or on GitHub) —
the terminal cannot render LaTeX, so this notebook is where the formulas live.

Reference document, not an exercise. Nothing to run except the last section.

| § | Topic |
|:--|:--|
| 1 | The P&L functional, and where it comes from |
| 2 | Why the cost sum runs to $n$ |
| 3 | Discounted units — the numéraire |
| 4 | Geometric Brownian motion, and the Itô correction |
| 5 | Black–Scholes reference values |
| 6 | The Whalley–Wilmott band, and the cube root |
| 7 | Risk measures and the training objective |
| 8 | Live check against the code |

---
# 1. The P&L functional

Trading dates $t_0 = 0 < t_1 < \dots < t_n = T$, evenly spaced with $\Delta t = T/n$.

| symbol | meaning | array shape |
|:--|:--|:--|
| $S_i$ | price at $t_i$ | `(n_paths, n_steps + 1)` |
| $\delta_i$ | position held over $[t_i, t_{i+1})$ | `(n_paths, n_steps)` |
| $Z$ | claim payoff at $T$ | `(n_paths,)` |
| $c$ | proportional cost rate | scalar |
| $p_0$ | premium received at $t_0$ | scalar |

$$
\boxed{\;\mathrm{PL}_T \;=\; p_0 \;-\; Z \;+\; \sum_{i=0}^{n-1} \delta_i\,(S_{i+1} - S_i) \;-\; \sum_{i=0}^{n} c\,S_i\,\lvert \delta_i - \delta_{i-1}\rvert\;}
$$

with $\delta_{-1} = 0$ (start flat) and $\delta_n = 0$ (liquidate at $T$).

We are **short** the claim ($-Z$) and **receive** the premium ($+p_0$).

## 1.1 Where it comes from

This is not a definition — it follows from the **self-financing** condition.

Let $X_i$ be the cash balance and $\delta_i$ the share holding at $t_i$. Just before
rebalancing at $t_{i+1}$, wealth is

$$
V_{i+1}^{-} \;=\; \delta_i S_{i+1} \;+\; X_i(1 + r\,\Delta t).
$$

Rebalancing swaps stock for cash and does **not** change wealth — except for the cost:

$$
X_{i+1} \;=\; X_i(1 + r\Delta t) \;-\; \underbrace{(\delta_{i+1} - \delta_i)S_{i+1}}_{\text{cash paid for shares}} \;-\; \underbrace{c\,S_{i+1}\lvert \delta_{i+1} - \delta_i \rvert}_{\text{cost, pure loss}}
$$

*"Rebalancing is wealth-neutral up to cost"* **is** the self-financing condition: no money
enters or leaves after inception.

Set $r = 0$ and telescope from $i = 0$ to $n$. The $\delta_i S_i$ terms cancel pairwise,
leaving the **discrete stochastic integral**

$$
\sum_{i=0}^{n-1} \delta_i (S_{i+1} - S_i) \;\;\longleftrightarrow\;\; \int_0^T \delta_t \, dS_t .
$$

## 1.2 Predictability — why $\delta_i$ multiplies $(S_{i+1} - S_i)$

$\delta_i$ is chosen at $t_i$ knowing $S_i$ but **not** $S_{i+1}$, and held across the gap.
So it earns the price change *over that gap*.

$$
\underbrace{n+1}_{\text{prices}} \quad\longrightarrow\quad \underbrace{n}_{\text{gaps}} \quad\longrightarrow\quad \underbrace{n}_{\text{decisions}}
$$

Pair $\delta_i$ with $(S_i - S_{i-1})$ instead and you are trading on an increment you have
already seen — a look-ahead. It raises no shape error. It presents as a hedge that works
implausibly well.

---
# 2. Why the cost sum runs to $n$

With $\delta_{-1} = \delta_n = 0$, the traded quantities are

$$
\underbrace{\delta_0 - 0}_{\text{open}},\;\; \delta_1 - \delta_0,\;\; \dots,\;\; \delta_{n-1} - \delta_{n-2},\;\; \underbrace{0 - \delta_{n-1}}_{\text{liquidate}}
$$

$$
n \text{ decisions} \quad\Longrightarrow\quad n+1 \text{ trades}
$$

**Accounting reason.** $\mathrm{PL}_T$ is a *cash* number. A share still held at $T$ is a
mark-to-market value; omitting the unwind asserts you can sell at mid.

**Incentive reason — the one that bites.** The terminal cost is the *only* term penalising
a large position at expiry. Drop it and a trained network learns it can carry an unbounded
hedge into maturity for free. The risk measure improves, nothing errors, and the strategy
is fictitious.

At 50bp on a call whose delta $\to 1$ in the money, the omitted charge is about
$c\,S_n \approx 0.5$ on spot 100 — a material slice of a typical premium.

---
# 3. Discounted units (the numéraire)

The telescoping in §1.1 needed $r = 0$. For $r \neq 0$ the cash account does not cancel.
Rather than track it, **change units**:

$$
\tilde S_i = e^{-r t_i} S_i, \qquad \tilde Z = e^{-rT} Z .
$$

Two facts combine.

**(a)** The discounted wealth of a self-financing strategy is the stochastic integral of
$\delta$ against the *discounted* price:
$\;\sum_i \delta_i(\tilde S_{i+1} - \tilde S_i)$.

**(b)** A cost paid at $t_i$ is $c\,S_i\lvert\Delta\delta_i\rvert$ in time-$t_i$ money, so

$$
e^{-r t_i}\cdot c\,S_i\,\lvert\Delta\delta_i\rvert \;=\; c\,\tilde S_i\,\lvert\Delta\delta_i\rvert .
$$

The cost discounts by the **same factor as the price it is charged on**. Nothing forced
that — a flat per-trade fee would not do it.

Therefore:

$$
\widetilde{\mathrm{PL}}_T \;=\; p_0 - \tilde Z + \sum_{i=0}^{n-1}\delta_i(\tilde S_{i+1} - \tilde S_i) - \sum_{i=0}^{n} c\,\tilde S_i \lvert \delta_i - \delta_{i-1}\rvert
$$

— **identical in form.** So `hedging_gains` and `transaction_costs` need no rate argument.
$p_0$ needs no adjustment either: it is received at $t_0$, so it is already time-0 money.

> **Assumes one rate for borrowing and lending.** Funding spreads or collateral would break
> the collapse into a single factor.

---
# 4. Geometric Brownian motion

$$
dS_t \;=\; \mu\,S_t\,dt \;+\; \sigma\,S_t\,dW_t
$$

Both terms scale with $S$: a \\$100 stock and a \\$200 stock move by the same **percentage**.

**Exact solution** (what `simulate_gbm` implements):

$$
\boxed{\;S_{i+1} \;=\; S_i \exp\!\Big( \big(\mu - \tfrac{\sigma^2}{2}\big)\Delta t \;+\; \sigma\sqrt{\Delta t}\,Z_i \Big), \qquad Z_i \sim \mathcal{N}(0,1)\;}
$$

## 4.1 The Itô correction $-\sigma^2/2$

The term people drop. By **Jensen's inequality**, for random $X$,

$$
\mathbb{E}\big[e^{X}\big] \;>\; e^{\mathbb{E}[X]} .
$$

For $X \sim \mathcal{N}(m, s^2)$ the gap is exact:
$\;\mathbb{E}[e^X] = e^{m + s^2/2}$.

So writing the exponent as $\mu\Delta t + \sigma\sqrt{\Delta t}\,Z$ gives

$$
\mathbb{E}[S_{i+1}] \;=\; S_i\,e^{\mu \Delta t}\cdot e^{\sigma^2 \Delta t / 2} \qquad \text{— too big.}
$$

Subtracting $\sigma^2/2$ cancels it exactly:

$$
\mathbb{E}[S_{i+1}] \;=\; S_i\,e^{\mu\Delta t}
$$

which is what *"drift $\mu$"* is supposed to mean. **Omit it and the rung-1 Monte Carlo call
price comes out high** — and it looks like a pricing bug, not a simulator bug.

## 4.2 Exact vs. Euler

$$
\text{Euler:}\quad S_{i+1} = S_i + \mu S_i \Delta t + \sigma S_i \sqrt{\Delta t}\,Z_i
$$

| | discretisation error | can go negative? |
|:--|:--|:--|
| Euler | $O(\Delta t)$ | yes |
| exact | **none** | no |

The exact form is the true conditional law of GBM for *any* $\Delta t$. So rung 1's residual
is pure Monte Carlo noise, shrinking as $1/\sqrt{n_{\text{paths}}}$. Euler adds a bias that
**does not shrink** when you add paths — a stubborn error immune to sampling harder.

## 4.3 Vectorising

$$
S_k \;=\; s_0 \exp\!\Big( \sum_{i<k} \big[(\mu - \tfrac{\sigma^2}{2})\Delta t + \sigma\sqrt{\Delta t} Z_i\big] \Big)
$$

The log-price is a **cumulative sum** — hence `tf.cumsum` along time, then `exp`, then
concatenate the $s_0$ column so it is bit-exact rather than a round trip through $e^0$.

---
# 5. Black–Scholes reference

$$
d_1 = \frac{\ln(S/K) + \big(r + \tfrac{\sigma^2}{2}\big)\tau}{\sigma\sqrt{\tau}}, \qquad d_2 = d_1 - \sigma\sqrt{\tau}, \qquad \tau = T - t
$$

$$
C = S\,\Phi(d_1) - K e^{-r\tau}\Phi(d_2), \qquad \Delta = \Phi(d_1), \qquad \Gamma = \frac{\varphi(d_1)}{S\sigma\sqrt{\tau}}
$$

with $\Phi$ the normal CDF and $\varphi$ its density.

**The gate.** Under GBM at zero cost, the risk-minimising hedge **is** $\Phi(d_1)$. A
correctly trained agent must reproduce it — rung 4, the single most important plot in the
project.

> Note it is $\approx$, not $=$. $\Phi(d_1)$ is optimal in *continuous* time. At finite $n$
> the truly risk-minimising discrete position differs slightly, and by how much depends on
> the risk measure. The agent solves the discrete problem — the one with no closed form.

---
# 6. The Whalley–Wilmott band

$$
\boxed{\;H = \left( \frac{3}{2}\cdot\frac{c\,S\,\Gamma^2 e^{-r(T-t)}}{\lambda} \right)^{1/3}\;}
$$

Hold while $\lvert \delta - \delta^{BS}\rvert \le H$. On exit, trade to the **nearest band
edge** — *not* back to $\delta^{BS}$.

$$
\begin{array}{c}
\delta^{BS} + H \quad\rule[0.5ex]{4cm}{0.4pt}\quad \text{edge}\\[2pt]
\text{no-transaction zone}\\[2pt]
\delta^{BS} - H \quad\rule[0.5ex]{4cm}{0.4pt}\quad \text{edge}
\end{array}
$$

The optimal control is **singular**: make the smallest trade that re-enters the
no-transaction region. Trading to the centre over-trades, weakens the baseline, and biases
the headline comparison toward deep hedging.

## 6.1 Why a cube root

Balance two competing effects:

$$
\text{total} \;\approx\; \underbrace{\frac{a\,c}{H}}_{\text{cost: wider band, fewer crossings}} \;+\; \underbrace{b\,H^2}_{\text{risk: wider band, further from } \delta^{BS}}
$$

$$
\frac{d}{dH}\Big[\frac{ac}{H} + bH^2\Big] = -\frac{ac}{H^2} + 2bH = 0 \quad\Longrightarrow\quad H^3 \propto c \quad\Longrightarrow\quad H \sim c^{1/3}
$$

Risk grows **quadratically** in $H$ while the saving grows only linearly, so the optimum
moves reluctantly.

$$
H \sim c^{1/3}, \qquad H \sim \Gamma^{2/3}, \qquad H \sim \lambda^{-1/3}
$$

Five times the cost widens the band by only $5^{1/3} \approx 1.71$. These three scalings are
the structural check on a learned agent: if its band does not respond to cost roughly as a
cube root, it has not found the right structure — regardless of its risk number.

---
# 7. The objective

$$
\min_\theta \; \rho\big(\mathrm{PL}_T(\theta)\big), \qquad \delta_i = \pi_\theta(I_i), \qquad I_i \supseteq (t_i,\, S_i,\, \delta_{i-1})
$$

**Entropic** (exponential utility), $\lambda > 0$:

$$
\rho_{\text{ent}}(X) = \frac{1}{\lambda}\log \mathbb{E}\big[e^{-\lambda X}\big]
$$

Implement with `reduce_logsumexp`; $\log(\mathrm{mean}(\exp(\cdot)))$ overflows.

**CVaR** at level $\alpha$, Rockafellar–Uryasev form (this is what makes it differentiable):

$$
\rho_{\text{CVaR}}(X) = \min_{w \in \mathbb{R}}\Big\{ w + \tfrac{1}{1-\alpha}\,\mathbb{E}\big[(-X - w)^{+}\big] \Big\}
$$

$w$ is a **trainable scalar optimised jointly with the weights**, not an inner loop.

**Mean–variance** (not coherent; included for comparability):

$$
\rho_{\text{mv}}(X) = -\mathbb{E}[X] + \tfrac{\lambda}{2}\mathrm{Var}(X)
$$

## 7.1 Cash-invariance

All three satisfy

$$
\rho(X + m) = \rho(X) - m .
$$

So $p_0$ merely **translates** the objective and cannot change $\delta^\star$. When a learned
policy appears indifferent to the premium, that is correct, not a bug.

## 7.2 Indifference price

$$
p_0 = \rho\big(\mathrm{PL}\text{ with claim}\big) - \rho\big(\mathrm{PL}\text{ without claim}\big)
$$

Two training runs. Reduces to the Black–Scholes price in the frictionless complete-market
limit — another checkable gate.

---
# 8. Live check against the code

Confirms the equations above match what `dhbench` actually computes.

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
sys.path.insert(0, "..")

import numpy as np, tensorflow as tf
from dhbench.pnl import hedging_gains, transaction_costs, turnover, terminal_pnl
from dhbench.worlds.gbm import simulate_gbm
from dhbench.baselines.bs_delta import bs_call_price, bs_delta, delta_hedge_positions

### §1–2 — the functional, on a path you can check by hand

$$
\text{gains} = 2(104-100) + 2(102-104) + (-1)(109-102) = 8 - 4 - 7 = -3
$$

In [ ]:
spot  = tf.constant([[100., 104., 102., 109.]])
delta = tf.constant([[2., 2., -1.]])

print("gains    ", hedging_gains(spot, delta).numpy(), "  hand: -3.00")
print("costs    ", transaction_costs(spot, delta, 0.01).numpy(), " hand: 6.15")
print("turnover ", turnover(delta).numpy(), "   hand: 6.00  (2+0+3+1, four trades)")

### §4.1 — the Itô correction

With the correction, $\mathbb{E}[S_T] = s_0 e^{\mu T}$. Without it, $s_0 e^{\mu T + \sigma^2 T/2}$.

In [ ]:
g = tf.random.Generator.from_seed(0)
s0, mu, sigma, T, n = 100.0, 0.05, 0.2, 1.0, 50

paths = simulate_gbm(200_000, n, s0, mu, sigma, T, g)

print("E[S_T] simulated ", float(tf.reduce_mean(paths[:, -1])))
print("s0*exp(mu*T)     ", s0*np.exp(mu*T), "  <- correct")
print("without Ito term ", s0*np.exp(mu*T + 0.5*sigma**2*T), "  <- what you'd get")
print("column 0 exact   ", bool(tf.reduce_all(paths[:, 0] == s0)))

### §5 — rung 1: Monte Carlo price vs. closed form

In [ ]:
K, r = 100.0, 0.0
paths = simulate_gbm(400_000, 50, s0, r, sigma, T, tf.random.Generator.from_seed(1))
mc = float(tf.reduce_mean(tf.maximum(paths[:, -1] - K, 0.0)))

print(f"Monte Carlo  {mc:.4f}")
print(f"closed form  {float(bs_call_price(s0, K, T, r, sigma)):.4f}")

### §5 — rung 2: the delta-hedge error shrinks as $n^{-1/2}$

$$
\mathrm{std}\big(\mathrm{PL}_T\big) \;\sim\; \frac{1}{\sqrt{n}}
$$

Watch the last column: quadrupling $n$ should roughly halve the error.

In [ ]:
premium = float(bs_call_price(s0, K, T, r, sigma))

print(f"{'n_steps':>8} {'mean PL':>10} {'std PL':>10} {'std*sqrt(n)':>13}")
for n_steps in [10, 40, 160, 640]:
    p = simulate_gbm(40_000, n_steps, s0, r, sigma, T, tf.random.Generator.from_seed(99))
    d = tf.constant(delta_hedge_positions(p.numpy(), K, T, r, sigma), dtype=tf.float32)
    pnl = terminal_pnl(p, d, tf.maximum(p[:, -1] - K, 0.0), 0.0, premium)
    m, s = float(tf.reduce_mean(pnl)), float(tf.math.reduce_std(pnl))
    print(f"{n_steps:>8} {m:>10.4f} {s:>10.4f} {s*np.sqrt(n_steps):>13.3f}")

The last column is roughly constant — that *is* the $1/\sqrt{n}$ law.

This is the strongest evidence available that the **simulator**, the **Black–Scholes
reference**, and the **P&L accounting** are mutually consistent. Any one could be wrong
alone; all three being wrong while still producing the right convergence rate is very
unlikely.

### §3 — numéraire: turnover is price-free, so $r$ cannot touch it

In [ ]:
payoff = tf.constant([9.])
print("PL at r=0.00 ", terminal_pnl(spot, delta, payoff, 0.01, 8.0).numpy())
print("PL at r=0.05 ", terminal_pnl(spot, delta, payoff, 0.01, 8.0, rate=0.05, maturity=1.0).numpy())
print("turnover     ", turnover(delta).numpy(), " <- unchanged: no price in it")